# Week 26, Day 2: Review: searching and sorting

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w026d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import bisect
xs = [3, 8, 8, 8, 15, 21]
print(bisect.bisect_left(xs, 8), bisect.bisect_right(xs, 8), bisect.bisect_left(xs, 9))

*Interactive figure on the website: Double n and watch each class. Slide n up and see where each one stops being practical..*

**Predict before you run:** what will this print?



In [ ]:
def a(xs):
    return xs[0] + xs[-1]

def b(xs):
    return sum(xs) / len(xs)

def c(xs):
    return any(x == y for i, x in enumerate(xs) for y in xs[i + 1:])

def d(n):
    steps = 0
    while n > 1:
        n //= 2
        steps += 1
    return steps

print(a([1, 2, 3]), b([1, 2, 3]), c([1, 2, 3, 2]), d(1_000_000))

> **Key idea.** Big-O is about growth. One loop is O(n), nested loops multiply, halving is O(log n), and the biggest term wins. Watch for hidden loops inside built-in operations, and say which case (best, worst) you mean.

*Interactive figure on the website: The highlighted range is where the target could still be. Each step halves it..*

**Predict before you run:** what will this print?



In [ ]:
def binary_search(xs, target):
    lo, hi = 0, len(xs) - 1
    steps = 0
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if xs[mid] == target:
            return mid, steps
        if xs[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1, steps

print(binary_search(list(range(0, 2000, 2)), 1234))
print(binary_search(list(range(0, 2000, 2)), 1235))

*Interactive figure on the website: Compare the counts on random, sorted and reversed data. Notice which sorts care about the starting order..*

**Predict before you run:** what will this print?



In [ ]:
def merge(a, b):
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    return out + a[i:] + b[j:]

print(merge([1, 4, 9], [2, 3, 10, 11]))

**Predict before you run:** what will this print?



In [ ]:
players = [("Ade", "U12", 31), ("Bo", "U14", 28), ("Cy", "U12", 31), ("Di", "U14", 35)]

by_tuple = sorted(players, key=lambda p: (p[1], -p[2], p[0]))
print([p[0] for p in by_tuple])

step = sorted(players, key=lambda p: p[0])
step = sorted(step, key=lambda p: p[2], reverse=True)
step = sorted(step, key=lambda p: p[1])
print([p[0] for p in step])

> **Key idea.** Binary search needs sorted data and takes O(log n). The simple sorts are O(n²); merge sort is O(n log n) and stable; quicksort is fast on average but O(n²) at worst; counting sort is O(n + k) for small whole numbers. Python's `sorted` is stable, so sort by a tuple key, or in passes from the least important key.

### Exercise: Name the growth

Give the Big-O of each function's running time, as one of `"O(1)"`, `"O(log n)"`, `"O(n)"`, `"O(n log n)"`, `"O(n^2)"`, where n is `len(xs)`.

In [ ]:
def f1(xs):
    return sorted(xs)[len(xs) // 2]

def f2(xs):
    seen = set()
    for x in xs:
        if x in seen:
            return True
        seen.add(x)
    return False

def f3(xs):
    out = []
    for x in xs:
        if x not in out:
            out.append(x)
    return out

def f4(xs):
    i, total = 1, 0
    while i < len(xs):
        total += xs[i]
        i *= 2
    return total

def f5(xs):
    return xs[len(xs) // 2] if xs else None

a = ___   # f1
b = ___   # f2
c = ___   # f3
d = ___   # f4
e = ___   # f5

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c, d, e) == (\"O(n log n)\", \"O(n)\", \"O(n^2)\", \"O(log n)\", \"O(1)\")")

### Exercise: The first match

Write `first_index(xs, target)` for a sorted list that may contain repeats: return the index of the **first** item equal to `target`, or -1. Use binary search (no `bisect`, no `index`, no `in`), so it stays O(log n).

In [ ]:
def first_index(xs, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    xs = [1, 3, 3, 3, 5, 8, 8, 13]\n    assert [first_index(xs, t) for t in (1, 3, 5, 8, 13, 0, 4, 20)] == [0, 1, 4, 5, 7, -1, -1, -1]\n    assert first_index([], 1) == -1 and first_index([7, 7, 7, 7], 7) == 0\ndef test_fast():\n    \"fast on a big list\"\n    import time\n    big = [i // 3 for i in range(300_000)]\n    t = time.time()\n    for target in range(0, 100_000, 97):\n        assert first_index(big, target) == target * 3\n    assert time.time() - t < 2\ndef test_own():\n    \"binary search written yourself\"\n    assert \"bisect\" not in source and \".index(\" not in source")

### Exercise: Insertion sort, counted

Write `insertion_sort(xs)` returning `(sorted_copy, shifts)`, where `shifts` counts how many times an item is moved one place to the right to make room. Don't change `xs`.

In [ ]:
def insertion_sort(xs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    xs = [5, 2, 4, 6, 1, 3]\n    assert insertion_sort(xs) == ([1, 2, 3, 4, 5, 6], 9) and xs == [5, 2, 4, 6, 1, 3]\n    assert insertion_sort([1, 2, 3]) == ([1, 2, 3], 0) and insertion_sort([3, 2, 1]) == ([1, 2, 3], 3)\n    assert insertion_sort([]) == ([], 0) and insertion_sort([2, 2, 1]) == ([1, 2, 2], 2)")

### Exercise: Merge sort with a key

Write `merge_sort(xs, key=lambda x: x)` returning a new list sorted by `key`, **stably**, using merge sort (no `sorted` or `.sort`).

In [ ]:
def merge_sort(xs, key=lambda x: x):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random as _r\ndef test_values():\n    \"values\"\n    assert merge_sort([5, 1, 4, 2, 3]) == [1, 2, 3, 4, 5] and merge_sort([]) == [] and merge_sort([1]) == [1]\n    words = [\"pear\", \"fig\", \"apple\", \"kiwi\", \"plum\", \"date\"]\n    assert merge_sort(words, key=len) == [\"fig\", \"pear\", \"kiwi\", \"plum\", \"date\", \"apple\"]\ndef test_random():\n    \"random lists, stably\"\n    rng = _r.Random(4)\n    for _ in range(50):\n        xs = [(rng.randint(0, 5), i) for i in range(rng.randint(0, 30))]\n        assert merge_sort(xs, key=lambda p: p[0]) == sorted(xs, key=lambda p: p[0])\ndef test_own():\n    \"merge sort written yourself\"\n    assert \"sorted(\" not in source and \".sort(\" not in source")

### Exercise: A league table

Each team is `(name, points, scored, conceded)`. Write `league(teams)` returning the names in table order: most points first, then the best goal difference (scored minus conceded), then most goals scored, then name A to Z.

In [ ]:
def league(teams):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"the table\"\n    teams = [(\"Rovers\", 10, 8, 5), (\"Athletic\", 10, 9, 6), (\"City\", 12, 7, 7), (\"United\", 10, 6, 3), (\"Albion\", 10, 8, 5)]\n    assert league(teams) == [\"City\", \"Athletic\", \"Albion\", \"Rovers\", \"United\"]\n    assert league([]) == []")

### Exercise: Counting sort

Write `counting_sort(marks)` for whole-number marks from 0 to 100, without comparing marks to each other: count each mark, then build the sorted list from the counts.

In [ ]:
def counting_sort(marks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random as _r\ndef test_values():\n    \"values\"\n    assert counting_sort([70, 40, 100, 0, 70]) == [0, 40, 70, 70, 100] and counting_sort([]) == []\n    rng = _r.Random(1)\n    xs = [rng.randint(0, 100) for _ in range(2000)]\n    assert counting_sort(xs) == sorted(xs)\ndef test_no_compare():\n    \"no sorted, no comparisons between marks\"\n    assert \"sorted(\" not in source and \".sort(\" not in source and \"min(\" not in source")

### Bug hunt: Stuck in the middle

This binary search sometimes never finishes. Find the case, and fix it so it always finishes, keeping it a binary search.

In [ ]:
def contains(xs, target):
    lo, hi = 0, len(xs) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if xs[mid] < target:
            lo = mid
        else:
            hi = mid
    return bool(xs) and xs[lo] == target

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"always finishes, and right\"\n    xs = [2, 4, 6, 8, 10]\n    for t in range(0, 12):\n        assert contains(xs, t) is (t in xs), t\n    assert contains([], 3) is False and contains([5], 5) is True and contains([1, 3], 3) is True")

### Bug hunt: Passes in the wrong order

The aim: runners sorted by club A to Z, and within each club, fastest time first. The passes are in the wrong order. Fix it (keep two stable passes).

In [ ]:
runners = [("Ada", "Harriers", 41.2), ("Bo", "Striders", 39.8), ("Cy", "Harriers", 38.5), ("Di", "Striders", 44.0), ("Ed", "Harriers", 41.2)]

def club_order(runners):
    step = sorted(runners, key=lambda r: r[1])
    step = sorted(step, key=lambda r: r[2])
    return [r[0] for r in step]

print(club_order(runners))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"by club, then time\"\n    assert club_order(runners) == [\"Cy\", \"Ada\", \"Ed\", \"Bo\", \"Di\"]\n    assert \"sorted\" in source and source.count(\"sorted(\") == 2")

### Bug hunt: Gone

This should print the scores from highest to lowest, and then the original list unchanged. It prints `None`. Fix it.

In [ ]:
scores = [41, 77, 63, 58]
ranked = scores.sort(reverse=True)
print(ranked)
print(scores)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both lines\"\n    expect_output(\"[77, 63, 58, 41]\\n[41, 77, 63, 58]\")")

### Puzzle: Binary search

Arrange the lines into a binary search that returns the index of `target`, or -1. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer without running anything.

In [ ]:
a = ___   # most steps binary search needs for 1,000,000 sorted items (a whole number)
b = ___   # an O(n^2) sort takes 1 second for 10,000 items: roughly how many seconds for 100,000?
c = ___   # True or False: quicksort is always O(n log n)
d = ___   # True or False: Python's sorted is stable
e = ___   # insertion sort's Big-O on data that's already sorted, written like "O(n)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 20 and b == 100 and c is False and d is True and e == \"O(n)\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the closest sum

Write `closest_sum(xs, target)` returning the sum of two items, at different positions in `xs`, that is closest to `target`. If a sum below and a sum above are equally close, return the smaller. `None` for fewer than two items. It must handle 100,000 items quickly, so trying every pair (O(n²)) is too slow.

In [ ]:
def closest_sum(xs, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random as _r\ndef slow(xs, target):\n    sums = [xs[i] + xs[j] for i in range(len(xs)) for j in range(i + 1, len(xs))]\n    return min(sums, key=lambda s: (abs(s - target), s)) if sums else None\ndef test_small():\n    \"small lists\"\n    assert closest_sum([1, 8, 3, 12, 5], 10) == 9 and closest_sum([4], 4) is None and closest_sum([], 1) is None\n    assert closest_sum([5, 5], 9) == 10 and closest_sum([-3, 9, 2, 7], 0) == -1 and closest_sum([1, 3], 3) == 4\n    assert closest_sum([2, 4, 6], 8) == 8 and closest_sum([1, 2, 10, 11], 7) == 3\n    rng = _r.Random(2)\n    for _ in range(500):\n        xs = [rng.randint(-20, 20) for _ in range(rng.randint(2, 9))]\n        t = rng.randint(-30, 30)\n        assert closest_sum(xs, t) == slow(xs, t), (xs, t)\ndef test_fast():\n    \"100,000 items\"\n    import time\n    rng = _r.Random(3)\n    xs = [rng.randint(0, 10**9) for _ in range(100_000)]\n    t = time.time()\n    closest_sum(xs, 10**9)\n    assert time.time() - t < 3")

## Warm-up

`bisect` is the standard library's binary search: `bisect_left` gives the first position where the value could go, `bisect_right` the position after any equal items. Both take O(log n) steps.

## Big-O

Big-O describes how the **number of steps grows** as the input grows, ignoring constant factors and small terms. It answers "what happens if the input is ten times bigger?"

| Big-O | Name | Doubling n... | Example |
|---|---|---|---|
| O(1) | constant | changes nothing | indexing a list, a dictionary lookup |
| O(log n) | logarithmic | adds one step | binary search |
| O(n) | linear | doubles the work | one loop over the data, `x in list` |
| O(n log n) | linearithmic | a bit more than doubles | merge sort, `sorted` |
| O(n²) | quadratic | quadruples | a loop inside a loop over the data; the simple sorts |
| O(2ⁿ) | exponential | squares it | plain tree recursion like Fibonacci |

To analyse code: a loop over n items costs n times its body; nested loops multiply; steps one after another add, and the biggest term wins; a loop that halves its range each time is O(log n).

`a` is O(1), `b` O(n), `c` O(n²) in the worst case, `d` O(log n): twenty halvings take a million down to 1.

Watch for hidden loops: `x in some_list`, `list.index`, `list.remove`, `list.insert(0, x)`, slicing and `sorted` all take time proportional to the list. `x in some_set` and `d[key]` are O(1) on average.

Big-O usually means the **worst case**. Insertion sort is O(n²) in the worst case (reversed data) but O(n) in the best (already sorted). **Space** counts the extra memory: merge sort needs O(n) extra; a recursive function uses one frame per level of calls.

## Searching

**Linear search** checks each item in turn: O(n), and works on any list. **Binary search** needs a **sorted** list: compare with the middle item, then throw away the half that can't contain the target. O(log n): about 20 steps for a million items, 30 for a billion.

Sorting first just to search once is a loss (O(n log n) to save O(n)), but if you'll search many times, sort once and binary search every time, or use a set.

## Sorting

| Sort | Idea | Worst | Best | Stable? |
|---|---|---|---|---|
| Selection | find the smallest remaining, swap it into place | O(n²) | O(n²) | no |
| Insertion | take the next item, shift it left into place | O(n²) | O(n) | yes |
| Bubble | swap neighbours that are out of order, repeat | O(n²) | O(n) with an early stop | yes |
| Merge | split in half, sort each, merge | O(n log n) | O(n log n) | yes |
| Quicksort | partition round a pivot, sort each side | O(n²) | O(n log n) | not usually |
| Counting | count each value, write them out in order | O(n + k) | O(n + k) | yes (the careful version) |
| Timsort (`sorted`) | finds runs, merges them | O(n log n) | O(n) | yes |

The `<=` (not `<`) makes merge sort **stable**: when two items are equal, the one from the left half, which came first, goes first.

Counting sort beats O(n log n) because it never compares items: it only works when the values are small whole numbers (marks out of 100, ages), where `k`, the range of values, isn't much bigger than n.

## Sorting by keys

A **stable** sort keeps items with equal keys in their original order. That's what lets you sort by several things: either use one key that's a **tuple**, or sort several times, **least important key first**.

Both give the players by age group, then highest score first, then by name. The negative score in the tuple sorts numbers high to low; for text, which can't be negated, sort in several stable passes instead. `reverse=True` keeps stability too.

## The classic mistakes

1. **Binary search on unsorted data**: it runs, and gives wrong answers.
2. **Off-by-one in binary search**: `lo = mid` instead of `lo = mid + 1` can loop for ever when `lo` and `hi` are next to each other.
3. **Hidden quadratic code**: `if x not in result_list: result_list.append(x)` in a loop is O(n²); a set makes it O(n).
4. **Sorting in passes in the wrong order**: the **last** pass is the most important key.
5. **`list.sort()` returns `None`**: `xs = xs.sort()` loses your list. Use `sorted(xs)` for a new list.
6. **Confusing best and worst cases**: "insertion sort is O(n)" is only true for data that's already sorted.

## Practice

## Quiz

## Daily challenge

## Before you go

Add today's surprises to your list. Then pick the sort you'd find hardest to write from memory, and write it from memory, with a test against `sorted` on random lists. Tomorrow: modules, the command line, Git, data formats and testing.